# Experiment 6 - K-Means and Hierarchical Clustering on Mall Customers

**Objective:** Apply K-Means and Hierarchical Clustering on the mall customer dataset and interpret customer segments.

**Dataset:** `Mall_Customers.csv`

## Step 1 - Import Libraries

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans, AgglomerativeClustering
from sklearn.metrics import silhouette_score
from scipy.cluster.hierarchy import dendrogram, linkage

sns.set_style("whitegrid")

## Step 2 - Load and Inspect the Dataset

In [ ]:
df = pd.read_csv("Mall_Customers.csv")

print(f"Dataset shape: {df.shape}")
print("\nColumn names:", df.columns.tolist())
df.head()

In [ ]:
df.info()
print("\nMissing values per column:")
print(df.isnull().sum())

## Step 3 - Select Features for Clustering

We use the two most common segmentation features:
- Annual Income (k$)
- Spending Score (1-100)

In [ ]:
X = df[["Annual Income (k$)", "Spending Score (1-100)"]]

plt.figure(figsize=(7, 5))
plt.scatter(X.iloc[:, 0], X.iloc[:, 1], c="gray", s=40)
plt.xlabel("Annual Income (k$)")
plt.ylabel("Spending Score (1-100)")
plt.title("Customer Distribution")
plt.show()

## Step 4 - K-Means Clustering

We first use the Elbow Method to find a suitable number of clusters.

In [ ]:
wcss = []
k_values = range(1, 11)

for k in k_values:
    km = KMeans(n_clusters=k, random_state=42, n_init=10)
    km.fit(X)
    wcss.append(km.inertia_)

plt.figure(figsize=(7, 5))
plt.plot(k_values, wcss, marker="o")
plt.xlabel("Number of clusters (k)")
plt.ylabel("WCSS (Inertia)")
plt.title("Elbow Method for K-Means")
plt.xticks(k_values)
plt.show()

In [ ]:
# Based on the elbow curve, choose k=5
kmeans = KMeans(n_clusters=5, random_state=42, n_init=10)
df["KMeans_Cluster"] = kmeans.fit_predict(X)

plt.figure(figsize=(8, 6))
scatter = plt.scatter(
    X.iloc[:, 0],
    X.iloc[:, 1],
    c=df["KMeans_Cluster"],
    cmap="tab10",
    s=50
)

centers = kmeans.cluster_centers_
plt.scatter(centers[:, 0], centers[:, 1], c="black", s=220, marker="X", label="Centroids")
plt.xlabel("Annual Income (k$)")
plt.ylabel("Spending Score (1-100)")
plt.title("K-Means Clusters (k=5)")
plt.legend()
plt.show()

In [ ]:
# Silhouette score for chosen K-Means model
kmeans_silhouette = silhouette_score(X, df["KMeans_Cluster"])
print(f"K-Means Silhouette Score (k=5): {kmeans_silhouette:.4f}")

## Step 5 - Hierarchical Clustering

We use Ward linkage and visualize customer grouping with a dendrogram.

In [ ]:
linked = linkage(X, method="ward")

plt.figure(figsize=(12, 5))
dendrogram(linked)
plt.title("Dendrogram (Ward Linkage)")
plt.xlabel("Customers")
plt.ylabel("Euclidean Distance")
plt.show()

In [ ]:
# Choose 5 clusters for comparison with K-Means
hc = AgglomerativeClustering(n_clusters=5, metric="euclidean", linkage="ward")
df["HC_Cluster"] = hc.fit_predict(X)

plt.figure(figsize=(8, 6))
plt.scatter(
    X.iloc[:, 0],
    X.iloc[:, 1],
    c=df["HC_Cluster"],
    cmap="tab10",
    s=50
)
plt.xlabel("Annual Income (k$)")
plt.ylabel("Spending Score (1-100)")
plt.title("Hierarchical Clusters (n_clusters=5)")
plt.show()

In [ ]:
hc_silhouette = silhouette_score(X, df["HC_Cluster"])
print(f"Hierarchical Silhouette Score (n_clusters=5): {hc_silhouette:.4f}")

## Step 6 - Compare Clustering Models

In [ ]:
comparison = pd.DataFrame({
    "Model": ["K-Means", "Hierarchical"],
    "No. of Clusters": [5, 5],
    "Silhouette Score": [kmeans_silhouette, hc_silhouette]
})

comparison["Silhouette Score"] = comparison["Silhouette Score"].round(4)
print(comparison.to_string(index=False))

## Conclusion

In this experiment, we applied two unsupervised clustering techniques to segment mall customers based on **Annual Income** and **Spending Score**.

- **K-Means Clustering:** We used the Elbow Method to select a suitable number of clusters (`k=5`), trained K-Means, and visualized customer groups with centroids.
- **Hierarchical Clustering:** We built a dendrogram using Ward linkage and then formed 5 agglomerative clusters for direct comparison.
- **Model Comparison:** We compared both methods using silhouette score to quantify cluster separation quality.

**Key takeaway:** Both K-Means and Hierarchical clustering can identify meaningful customer segments in this dataset. K-Means is generally simpler and faster for larger datasets, while Hierarchical clustering provides richer structural insight through the dendrogram.